# Assignment 2 — The Dirty Ledger
## Data loading, cleaning and data quality

This notebook works on a copy of `teaching_prices_messy.csv`. It preserves questionable observations and documents every change. Put the CSV in the same folder as this notebook before running it.

## 1. Load the original data and inspect it

In [ ]:
import pandas as pd

df = pd.read_csv("teaching_prices_messy.csv", dtype="string")
print("Shape:", df.shape)
display(df.head())
print("\\nData types:")
print(df.dtypes)


## 2. Diagnose data quality before changing anything

In [ ]:
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))

print("Exact duplicate rows:", df.duplicated().sum())
print("\\nUnique values in item:")
print(df["item"].value_counts(dropna=False))

print("\\nOriginal price values that do not parse directly as numbers:")
direct_numeric = pd.to_numeric(df["price"], errors="coerce")
display(df.loc[df["price"].notna() & direct_numeric.isna(), ["date", "item", "city", "price"]])

print("\\nNumeric prices after removing commas:")
price_numeric_preview = pd.to_numeric(
    df["price"].str.replace(",", "", regex=False), errors="coerce"
)
print(price_numeric_preview.describe())
display(df.loc[price_numeric_preview > 500, ["date", "item", "city", "price"]])


## 3. Cleaning plan
- Remove one exact duplicate only after checking all columns match.
- Standardize the one `oil b` spelling to `Oil B`, while preserving the original item value.
- Convert comma-formatted prices to numeric values.
- Keep the missing price missing.
- Keep 1250 and 9999, but flag them for human review. The `> 500` threshold is a screening choice based on the visible dataset pattern, not an official rule.

In [ ]:
# Work on a copy and preserve original values for the audit trail.
clean = df.copy()
clean["item_original"] = clean["item"]
clean["price_original"] = clean["price"]

# Standardize whitespace and the one observed capitalization inconsistency.
clean["item"] = clean["item"].str.strip()
clean["item"] = clean["item"].str.replace(
    r"(?i)^oil b$", "Oil B", regex=True
)

# Parse dates and prices. Removing commas converts the text "1,250" to 1250.
clean["date"] = pd.to_datetime(clean["date"], errors="coerce")
clean["price"] = pd.to_numeric(
    clean["price"].str.replace(",", "", regex=False), errors="coerce"
)

# Mark exact duplicates before removing them.
clean["is_duplicate"] = clean.duplicated(
    subset=["date", "item_original", "city", "price_original", "unit", "data_label"],
    keep="first"
)
duplicate_count = int(clean["is_duplicate"].sum())
clean = clean.loc[~clean["is_duplicate"]].copy()

# Add transparent review flags; do not delete or impute questionable observations.
clean["missing_price"] = clean["price"].isna()
clean["review_price"] = clean["price"].gt(500)

print("Exact duplicates removed:", duplicate_count)
print("Cleaned shape:", clean.shape)


## 4. Before-and-after verification

In [ ]:
before_numeric = pd.to_numeric(
    df["price"].str.replace(",", "", regex=False), errors="coerce"
)

comparison = pd.DataFrame({
    "check": [
        "Rows",
        "Exact duplicate rows",
        "Missing prices",
        "Numeric prices",
        "Minimum numeric price",
        "Median numeric price",
        "Maximum numeric price"
    ],
    "before": [
        len(df),
        int(df.duplicated().sum()),
        int(before_numeric.isna().sum()),
        int(before_numeric.notna().sum()),
        before_numeric.min(),
        before_numeric.median(),
        before_numeric.max()
    ],
    "after": [
        len(clean),
        int(clean.duplicated(subset=["date", "item_original", "city", "price_original", "unit", "data_label"]).sum()),
        int(clean["price"].isna().sum()),
        int(clean["price"].notna().sum()),
        clean["price"].min(),
        clean["price"].median(),
        clean["price"].max()
    ]
})
display(comparison)

print("Rows needing price review:")
display(clean.loc[clean["review_price"], [
    "date", "item", "city", "price_original", "price", "review_price"
]])

print("Rows with missing prices:")
display(clean.loc[clean["missing_price"], [
    "date", "item", "city", "price_original", "price"
]])

print("Standardized item names:")
print(clean["item"].value_counts())


## 5. Save the cleaned working copy
The output retains the original text fields (`item_original`, `price_original`) so a reviewer can trace changes. The original CSV is not overwritten.

In [ ]:
clean.to_csv("dirty_ledger_cleaned.csv", index=False)
print("Saved dirty_ledger_cleaned.csv")
